# Neural Turing Machines on the copy task

Reproduction of [Graves, Wayne & Danihelka (2014)](https://arxiv.org/abs/1410.5401): an LSTM baseline and the NTM with a feed-forward and an LSTM controller.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then run the cells in order.
Leave the tab open and the laptop awake while it trains.

## 1. Get the code

Safe to re-run: it wipes any old clone and starts clean.

In [ ]:
%cd /content
!rm -rf neural-turing-machines
!git clone -q https://github.com/mgupta8143/neural-turing-machines.git
%cd neural-turing-machines
!git log --oneline -1

## 2. Save results to Google Drive

Optional but recommended: figures and results are copied here every 15 minutes, so a Colab disconnect costs you nothing.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Train everything

Starts all three models at once and keeps every figure up to date while they run:

| Model | Device | Settings | Time to converge |
|---|---|---|---|
| `ntm-ff` | CPU | batch 16, lr 1e-4 | ~1 hour |
| `ntm-lstm` | CPU | batch 16, lr 1e-4 | ~1 hour |
| `lstm` | GPU | the paper's batch 1, lr 3e-5 | ~6 hours |

The NTMs run on the CPU on purpose: their per-timestep loop of small operations is faster there than on a GPU, which spends its time launching kernels. Every 15 minutes the script redraws all the figures from the latest checkpoints and copies them to Drive.

In [ ]:
!nohup bash overnight.sh > overnight.log 2>&1 &
!sleep 60 && pgrep -af 'main.py train'

**Check the cell above printed three lines**, one per model. If it printed nothing, training did not start — re-run it. Never run `pkill` unless you mean to stop everything.

In [ ]:
!tail -n 2 train_ntm-ff.log train_ntm-lstm.log train_lstm.log

## 4. Progress

Re-run this whenever you like. Cost is in bits per sequence: 84 is random guessing, 0 is a perfect copy.

In [ ]:
!tail -n 3 overnight.log train_ntm-ff.log train_ntm-lstm.log train_lstm.log

## 5. The figures

- **Learning curve** — the paper's Figure 3.
- **Generalisation** — the paper's Figure 5: lengths 10, 20, 30, 50 and 120, trained only on 1–20.
- **Memory use** — the paper's Figure 6, NTM only: inputs, adds and write weightings on the left; outputs, reads and read weightings on the right. A diagonal stripe means the head is stepping one memory location at a time, which is the copy algorithm.

In [ ]:
import os
from IPython.display import Image, display

figures = ['ntm-ff_learning_curve', 'ntm-ff_generalisation',
           'ntm-ff_memory_length20', 'ntm-ff_memory_length40',
           'ntm-lstm_learning_curve', 'ntm-lstm_generalisation',
           'ntm-lstm_memory_length20', 'ntm-lstm_memory_length40',
           'lstm_learning_curve', 'lstm_generalisation']
for name in figures:
    path = f'figures/{name}.png'
    if os.path.exists(path):
        print(name)
        display(Image(path))

## 6. Try your own sequence

Each item is 8 bits. Set `random_length` to a number instead to use a random sequence of that length — try something well past 20, where the LSTM fails and the NTM should not.

In [ ]:
MODEL = 'ntm-ff'
vectors = '10110010 01100101 11110000 00001111'
random_length = None  # e.g. 40

if random_length:
    !python main.py try --model {MODEL} --random {random_length}
else:
    !python main.py try --model {MODEL} {vectors}
from IPython.display import Image, display
display(Image(f'figures/{MODEL}_try.png'))

## 7. Download everything

Figures plus the raw training logs.

In [ ]:
!zip -qr results.zip figures results/copy/*/log.csv
from google.colab import files
files.download('results.zip')

## Stopping

Only when you actually want to stop training.

In [ ]:
!pkill -f 'main.py train'; pkill -f overnight.sh